Deep Q-Network (DQN) Maze Solver — Keras
=========================================
Problem: Navigate a grid maze from START (top-left) to GOAL (bottom-right),
         avoiding walls, in as few steps as possible.

State:   (row, col) encoded as a one-hot vector of length ROWS*COLS

Actions: UP, DOWN, LEFT, RIGHT  (4 discrete actions)

Reward:  +10  on reaching GOAL
         -1   each step (encourages shortest path)
         -5   on hitting a wall (agent stays in place)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from collections import deque
import random

In [ ]:
# ─────────────────────────────────────────
# 1. HYPERPARAMETERS
# ─────────────────────────────────────────
ROWS, COLS = 8, 8
N_ACTIONS = 4          # UP, DOWN, LEFT, RIGHT
N_STATES = ROWS * COLS

EPISODES = 200
MAX_STEPS = 50        # max moves per episode

MEMORY_SIZE = 1000
BATCH_SIZE = 32
GAMMA = 0.95       # discount — maze requires planning ahead
LR = 5e-4
TARGET_UPDATE = 20         # sync target network every N episodes

EPSILON_START = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.992

SOLVE_SCORE = 10.0        # avg reward over 50 eps to declare "solved"

# Action index → (row_delta, col_delta)
ACTIONS = {
    0: (-1,  0),   # UP
    1: ( 1,  0),   # DOWN
    2: ( 0, -1),   # LEFT
    3: ( 0,  1),   # RIGHT
}

In [ ]:
# ─────────────────────────────────────────
# 2. MAZE DEFINITION
# ─────────────────────────────────────────
# 0 = open, 1 = wall
MAZE = np.array([
    [0, 0, 0, 1, 0, 0, 0, 0],
    [1, 1, 0, 1, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0],
    [1, 1, 0, 1, 1, 1, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 1, 0, 0],
], dtype=np.int32)

START = (0, 0)
GOAL = (7, 7)

In [ ]:
# ─────────────────────────────────────────
# 3. MAZE ENVIRONMENT
# ─────────────────────────────────────────
class MazeEnv:
    """
    Grid world environment.

    State encoding: one-hot vector of length ROWS*COLS.
    Position (r, c) → index r*COLS + c → one-hot[index] = 1.

    This lets the neural network distinguish every cell uniquely
    without implying any spatial ordering between cells.
    """
    def __init__(self):
        self.reset()

    def reset(self) -> np.ndarray:
        self.pos = START
        return self._encode(self.pos)

    def step(self, action: int):
        dr, dc = ACTIONS[action]
        r, c = self.pos
        nr, nc = r + dr, c + dc
        # Out of bounds or wall → stay, penalise
        if not (0 <= nr < ROWS and 0 <= nc < COLS) or MAZE[nr, nc] == 1:
            reward = -5.0
            done = False
        else:
            self.pos = (nr, nc)
            if self.pos == GOAL:
                reward = 10.0
                done = True
            else:
                reward = -1.0   # step cost: shortest path is best path
                done = False
        return self._encode(self.pos), reward, done

    def _encode(self, pos) -> np.ndarray:
        """One-hot encode (row, col) → vector of length ROWS*COLS."""
        state = np.zeros(N_STATES, dtype=np.float32)
        state[pos[0] * COLS + pos[1]] = 1.0
        return state

In [ ]:
# ─────────────────────────────────────────
# 4. Q-NETWORK (Keras)
# ─────────────────────────────────────────
def build_model() -> keras.Model:
    """
    Input:  one-hot state vector  (N_STATES,)
    Output: Q-values for each action  (N_ACTIONS,)

    Larger hidden layers than the bandit because the maze
    requires the network to learn spatial relationships.
    """
    model = keras.Sequential([
        keras.layers.Input(shape=(N_STATES,)),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(64,  activation="relu"),
        keras.layers.Dense(N_ACTIONS, activation="linear"),
    ], name="MazeQNet")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LR), loss="mse",)
    return model

In [ ]:
# ─────────────────────────────────────────
# 5. REPLAY BUFFER
# ─────────────────────────────────────────
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states = np.array([b[0] for b in batch], dtype=np.float32)
        actions = np.array([b[1] for b in batch], dtype=np.int32)
        rewards = np.array([b[2] for b in batch], dtype=np.float32)
        next_states = np.array([b[3] for b in batch], dtype=np.float32)
        dones = np.array([b[4] for b in batch], dtype=np.float32)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

In [ ]:
# ─────────────────────────────────────────
# 6. DQN AGENT
# ─────────────────────────────────────────
class MazeDQNAgent:
    def __init__(self):
        self.epsilon = EPSILON_START
        self.q_net = build_model()
        self.target_net = build_model()
        self.target_net.set_weights(self.q_net.get_weights())
        self.memory = ReplayBuffer(MEMORY_SIZE)

    def act(self, state: np.ndarray) -> int:
        if random.random() < self.epsilon:
            return random.randrange(N_ACTIONS)
        q_values = self.q_net.predict(state[np.newaxis], verbose=0)[0]
        return int(np.argmax(q_values))

    def learn(self):
        if len(self.memory) < BATCH_SIZE:
            return
        states, actions, rewards, next_states, dones = self.memory.sample(BATCH_SIZE)
        # Target: r  +  γ · max_a' Q_target(s', a')   [masked at terminal]
        next_q = self.target_net.predict(next_states, verbose=0)
        max_next_q = np.max(next_q, axis=1)
        td_targets = rewards + GAMMA * max_next_q * (1 - dones)
        # Only update Q for the action taken; leave others unchanged
        current_q = self.q_net.predict(states, verbose=0)
        for i, action in enumerate(actions):
            current_q[i, action] = td_targets[i]
        self.q_net.fit(states, current_q, verbose=0, epochs=1, batch_size=BATCH_SIZE)

    def update_target(self):
        self.target_net.set_weights(self.q_net.get_weights())

    def decay_epsilon(self):
        self.epsilon = max(EPSILON_MIN, self.epsilon * EPSILON_DECAY)

In [ ]:
# ─────────────────────────────────────────
# 7. TRAINING LOOP
# ─────────────────────────────────────────
def print_maze(path=None):
    """Pretty-print the maze, optionally overlaying the agent's path."""
    symbols = {0: "·", 1: "█"}
    grid = [[symbols[MAZE[r, c]] for c in range(COLS)] for r in range(ROWS)]
    grid[START[0]][START[1]] = "S"
    grid[GOAL[0]][GOAL[1]] = "G"
    if path:
        for (r, c) in path:
            if (r, c) not in (START, GOAL):
                grid[r][c] = "○"
    for row in grid:
        print("  " + " ".join(row))


def greedy_path(agent: MazeDQNAgent, env: MazeEnv):
    """Run one greedy episode, return list of positions visited."""
    state = env.reset()
    path = [env.pos]
    for _ in range(MAX_STEPS):
        q = agent.q_net.predict(state[np.newaxis], verbose=0)[0]
        action = int(np.argmax(q))
        state, _, done = env.step(action)
        path.append(env.pos)
        if done:
            break
    return path


def train():
    env = MazeEnv()
    agent = MazeDQNAgent()
    scores = deque(maxlen=10)
    print("Maze layout (S=start, G=goal, █=wall):")
    print_maze()
    print()
    for episode in range(1, EPISODES + 1):
        state = env.reset()
        total_reward = 0.0
        for _ in range(MAX_STEPS):
            action = agent.act(state)
            next_state, reward, done = env.step(action)
            agent.memory.push(state, action, reward, next_state, float(done))
            agent.learn()
            state = next_state
            total_reward += reward
            if done:
                break
        agent.decay_epsilon()
        scores.append(total_reward)
        if episode % TARGET_UPDATE == 0:
            agent.update_target()
        if episode % 10 == 0:
            avg = np.mean(scores)
            print(f"Episode {episode:5d} | Avg reward (10): {avg:7.2f} | ε: {agent.epsilon:.3f}")
            if avg >= SOLVE_SCORE:
                print(f"\nSolved at episode {episode}!")
                break
    # ── Show learned path ─────────────────
    print("\nLearned path (greedy policy):")
    path = greedy_path(agent, MazeEnv())
    print_maze(path)
    reached = path[-1] == GOAL
    print(f"\n{'Reached goal' if reached else 'Did not reach goal'} "
          f"in {len(path)-1} steps.")
    return agent

if __name__ == "__main__":
    train()

Maze layout (S=start, G=goal, █=wall):
  S · · █ · · · ·
  █ █ · █ · █ █ ·
  · · · · · · █ ·
  · █ █ █ █ · █ ·
  · · · · █ · · ·
  █ █ · █ █ █ █ ·
  · · · █ · · · ·
  · █ · · · █ · G

Episode    10 | Avg reward (10): -151.60 | ε: 0.923
Episode    20 | Avg reward (10): -149.20 | ε: 0.852
Episode    30 | Avg reward (10): -143.07 | ε: 0.786


KeyboardInterrupt: 

Simple Q-Learning Hallway — OOP Style
======================================

World:  [S] [ ] [ ] [ ] [G]

         0   1   2   3   4

Three classes:
  - Hallway   : the environment (world + rules)
  - Agent     : the learner    (Q-table + decisions)
  - Trainer   : the training loop + reporting

In [10]:
import random

In [15]:
# ─────────────────────────────────────────
# 1. ENVIRONMENT
# ─────────────────────────────────────────
class Environment:
    """
    A 5-cell corridor. The agent starts at cell 0 and must reach cell 4.
    Knows nothing about the agent — just enforces the rules of the world.
    """
    N_CELLS = 5
    GOAL = 4
    START = 0

    def __init__(self):
        return

    def reset(self) -> int:
        """Put the agent back at the start. Returns starting state."""
        self.state = self.START
        return self.state

    def step(self, action: int) -> tuple[int, float, bool]:
        """
        Apply action (0=left, 1=right).
        Returns (next_state, reward, done).
        """
        if action == 0:
            self.state = max(0, self.state - 1)
        else:
            self.state = min(self.N_CELLS - 1, self.state + 1)
        reached_goal = self.state == self.GOAL
        reward = 10.0 if reached_goal else -1.0
        return self.state, reward, reached_goal

In [16]:
# ─────────────────────────────────────────
# 2. AGENT
# ─────────────────────────────────────────
class Agent:
    """
    Learns via Q-learning. Maintains a Q-table and an epsilon for exploration.
    Knows nothing about the environment's rules — only sees states and rewards.
    """
    N_ACTIONS = 2   # 0 = left, 1 = right

    def __init__(self, n_states: int, lr: float, gamma: float, epsilon: float, eps_decay: float, eps_min: float):
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon
        self.eps_decay = eps_decay
        self.eps_min = eps_min
        # Q-table: rows = states, cols = actions, all start at 0
        self.Q = [[0.0] * self.N_ACTIONS for _ in range(n_states)]
        return

    def choose_action(self, state: int) -> int:
        """Epsilon-greedy: explore randomly OR exploit best known action."""
        if random.random() < self.epsilon:
            return random.randint(0, self.N_ACTIONS - 1)
        return self.Q[state].index(max(self.Q[state]))

    def update(self, state: int, action: int, reward: float, next_state: int, done: bool):
        """Apply the Q-learning update rule."""
        best_next = 0.0 if done else max(self.Q[next_state])
        td_target = reward + self.gamma * best_next
        td_error = td_target - self.Q[state][action]
        self.Q[state][action] += self.lr * td_error
        return

    def decay_epsilon(self):
        """Reduce exploration rate after each episode."""
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)
        return

    def best_action(self, state: int) -> int:
        """Pure greedy — no randomness. Used after training."""
        return self.Q[state].index(max(self.Q[state]))

In [17]:
# ─────────────────────────────────────────
# 3. TRAINER
# ─────────────────────────────────────────
class Trainer:
    """
    Runs episodes, wires the Agent and Environment together,
    and reports results.
    """
    ACTION_SYMBOLS = ["←", "→"]

    def __init__(self, env: Environment, agent: Agent, episodes: int, max_steps: int):
        self.env = env
        self.agent = agent
        self.episodes = episodes
        self.max_steps = max_steps
        self.wins = 0
        return

    def run(self):
        print("Training...\n")
        for episode in range(1, self.episodes + 1):
            self._run_episode()
            self.agent.decay_epsilon()
            if episode % 50 == 0:
                print(f"Episode {episode:3d} | "
                      f"wins: {self.wins:3d} | "
                      f"ε = {self.agent.epsilon:.2f}")
        self._report()
        return

    def _run_episode(self):
        state = self.env.reset()
        for _ in range(self.max_steps):
            action = self.agent.choose_action(state)
            next_state, reward, done = self.env.step(action)
            self.agent.update(state, action, reward, next_state, done)
            state = next_state
            if done:
                self.wins += 1
                break
        return

    def _report(self):
        print("\n── Learned Q-table ──────────────────────")
        print(f"{'Cell':<6} {'← left':>8} {'→ right':>8}   best")
        print("-" * 38)
        for s in range(self.env.N_CELLS):
            left, right = self.agent.Q[s]
            best_sym = self.ACTION_SYMBOLS[self.agent.best_action(s)]
            label = " (start)" if s == self.env.START else " (goal) " if s == self.env.GOAL else ""
            print(f"  {s}{label:<9} {left:>7.2f}  {right:>7.2f}   {best_sym}")
        print("\n── Greedy path (no exploration) ─────────")
        path = self._greedy_path()
        print(" → ".join(f"cell {s}" for s in path))
        won = path[-1] == self.env.GOAL
        print(f"\nReached goal in {len(path)-1} step(s). {'✓' if won else '✗'}")
        return

    def _greedy_path(self) -> list[int]:
        state = self.env.reset()
        path = [state]
        for _ in range(self.env.N_CELLS * 2):
            if state == self.env.GOAL:
                break
            action = self.agent.best_action(state)
            state, _, done = self.env.step(action)
            path.append(state)
            if done:
                break
        return path

In [18]:
# ─────────────────────────────────────────
# 4. MAIN
# ─────────────────────────────────────────
if __name__ == "__main__":
    env = Environment()
    agent = Agent(
        n_states = Environment.N_CELLS,
        lr = 0.5,
        gamma = 0.9,
        epsilon = 1.0,
        eps_decay = 0.98,
        eps_min = 0.05,
    )
    trainer = Trainer(env=env, agent=agent, episodes=200, max_steps=20)
    trainer.run()

Training...

Episode  50 | wins:  47 | ε = 0.36
Episode 100 | wins:  97 | ε = 0.13
Episode 150 | wins: 147 | ε = 0.05
Episode 200 | wins: 197 | ε = 0.05

── Learned Q-table ──────────────────────
Cell     ← left  → right   best
--------------------------------------
  0 (start)     3.12     4.58   →
  1             3.12     6.20   →
  2             4.58     8.00   →
  3             6.20    10.00   →
  4 (goal)      0.00     0.00   ←

── Greedy path (no exploration) ─────────
cell 0 → cell 1 → cell 2 → cell 3 → cell 4

Reached goal in 4 step(s). ✓
